# 🧭 MazeGPT — 反应式 2D 迷宫导航 (Google Colab 纯 RL 运行手册)

> **核心任务**：智能体每步仅观察四周 4 格（路 `.` / 墙 `#`），输出 `U/D/L/R`；撞墙原地停留；纯稀疏到达奖励，无任何 BFS 预训练。
> **核心结论**：Transformer 通过 GRPO 在 120 步内即可学会导航（到达率 83.3%），而 RNN/GRU 即使给予 5 倍预算（300步）也完全学不会。

## 步骤 1：挂载 Google Drive 并配置工作区

In [ ]:
from google.colab import drive
import os, sys

try:
    drive.mount('/content/drive')
    DRIVE_WORKSPACE = '/content/drive/MyDrive/maze_workspace'
    os.makedirs(f'{DRIVE_WORKSPACE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_WORKSPACE}/runs', exist_ok=True)
    print(f'✓ Google Drive 工作区就绪: {DRIVE_WORKSPACE}')
except Exception as e:
    print(f'Drive 挂载提示: {e}')
    DRIVE_WORKSPACE = None

## 步骤 2：环境准备与 Hugging Face 迷宫代码克隆

In [ ]:
!pip install -q torch openpyxl huggingface_hub matplotlib pandas

%cd /content
if not os.path.exists('/content/maze-transformer'):
    print('正在从 Hugging Face 克隆迷宫仓库...')
    !git clone https://huggingface.co/Hana-ame/maze-transformer /content/maze-transformer

%cd /content/maze-transformer
if '/content/maze-transformer' not in sys.path:
    sys.path.insert(0, '/content/maze-transformer')
print('✓ 迷宫导航环境与 Python 路径就绪')

## 步骤 3：从 Hugging Face 下载迷宫预训练模型 (.pt)

In [ ]:
import os, torch
from huggingface_hub import hf_hub_download

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)

REPO_ID = 'Hana-ame/maze-transformer'
CKPT_NAME = 'maze_grpo_final.pt'
os.makedirs('checkpoints', exist_ok=True)
local_path = f'checkpoints/{CKPT_NAME}'

print(f'正在从 Hugging Face 下载迷宫主模型: {CKPT_NAME} ...')
try:
    hf_hub_download(repo_id=REPO_ID, filename=f'checkpoints/{CKPT_NAME}', local_dir='.', token=hf_token)
    print(f'✓ 迷宫权重下载完成: {local_path} ({os.path.getsize(local_path)/1024:.1f} KB)')
    if DRIVE_WORKSPACE:
        !cp {local_path} {DRIVE_WORKSPACE}/checkpoints/{CKPT_NAME}
except Exception as e:
    print(f'下载提示: {e}，将直接进行从头 RL 训练。')

## 步骤 4：生成迷宫训练配置文件 (`maze_config.json`)

In [ ]:
import json

maze_cfg = {
    'layers': 2,                  # 2层 Transformer 结构最优
    'd': 64,                      # 隐藏通道宽度 64
    'heads': 4,                   # 4 头精确对应上下左右 4 格视场
    'steps': 120,                 # 训练步数 (纯RL 120步收敛)
    'batch_size': 6,              # 轨迹批量
    'lr': 3e-4,                   # 强化学习最佳 LR
    'min_size': 5,                # 最小迷宫尺寸
    'max_size': 9,                # 最大迷宫尺寸
    'single': True,               # 单轨迹训练
    'datasource': {
        'type': 'random_perfect_maze',
        'observation': 'forced_obs_4cell',
        'reward': 'sparse_goal_reach'
    }
}

with open('maze_config.json', 'w', encoding='utf-8') as f:
    json.dump(maze_cfg, f, indent=2)

print('✓ 迷宫配置文件已生成 maze_config.json:')
print(json.dumps(maze_cfg, indent=2))

## 步骤 5：拉起纯 RL 强化学习训练 (`train.py --config maze_config.json`)

In [ ]:
# 运行纯 RL 训练并观察 solvability 与到达率实时攀升
!python -m maze_transformer.train --config maze_config.json

## 步骤 6：迷宫多尺寸导航能力评估 (5x5 ~ 9x9)

In [ ]:
# 运行独立评测基准，输出各尺寸迷宫求解率
!python -m maze_transformer.bench || true

## 步骤 7：迷宫策略 INT8 动态量化评测

In [ ]:
# 验证离散动作策略在 INT8 量化下的稳定性
!python -m maze_transformer.quantize --checkpoint checkpoints/maze_grpo_final.pt

## 步骤 8：将产物归档至 Google Drive

In [ ]:
if DRIVE_WORKSPACE:
    !cp -ru runs/ {DRIVE_WORKSPACE}/runs/ || true
    !cp -ru checkpoints/ {DRIVE_WORKSPACE}/checkpoints/ || true
    print(f'✓ 迷宫模型与日志已同步备份至 Google Drive: {DRIVE_WORKSPACE}')